In [ ]:
pip install catboost

# XGBOOST 

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

df = pd.read_csv('../data/processed/telco_features.csv')  # Your new dataset


target_col = 'Churn'
feature_cols = [col for col in df.columns if col not in [target_col]]

# 3. Prepare X and y
X = df[feature_cols]
y = df[target_col]
# Your current one-hot encoded data is PERFECT for XGBoost
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model = XGBClassifier(
    n_estimators=1000,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc'
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=0
)

y_proba = model.predict_proba(X_test)[:, 1]
print(f"XGBoost AUC: {roc_auc_score(y_test, y_proba):.3f}")

# CATBOOST 

In [ ]:
import pandas as pd
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, classification_report

df_original_original = pd.read_csv('../data/processed/cleaned_churn.csv')  # ← Different file!

In [ ]:
# Load ORIGINAL data (before feature engineering one-hot encoding)

# Keep categorical columns as TEXT
categorical_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod'
]

# Verify they're text, not one-hot
print("Sample categorical values:")
for col in categorical_cols[:]:
    print(f"{col}: {df_original_original[col].unique()[:5]}")

# Now CatBoost will work!
feature_cols = [col for col in df_original_original.columns if col not in ['Churn', 'customerID']]
X = df_original_original[feature_cols]
y = df_original_original['Churn']

cat_features_indices = [i for i, col in enumerate(feature_cols) if col in categorical_cols]
print(f"\n✅ CatBoost will handle {len(cat_features_indices)} categorical columns natively")

# Train CatBoost
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
train_pool = Pool(X_train, y_train, cat_features=cat_features_indices)
test_pool = Pool(X_test, y_test, cat_features=cat_features_indices)

model = CatBoostClassifier(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    l2_leaf_reg=3,
    cat_features=cat_features_indices,
    random_state=42,
    verbose=0
)

model.fit(train_pool, eval_set=test_pool, early_stopping_rounds=50, use_best_model=True)

y_proba = model.predict_proba(test_pool)[:, 1]
print(f"\n🎯 CatBoost AUC (Native Categorical): {roc_auc_score(y_test, y_proba):.3f}")

# feature eng 

In [ ]:
# Tenure group
bins = [0, 12, 24, 48, 60, float('inf')]
labels = ['0-12', '12-24', '24-48', '48-60', '60-72']
df_original_original['tenure_group'] = pd.cut(df_original_original['tenure'], bins=bins, labels=labels)

# Average monthly charge
df_original_original['avg_monthly_charge'] = df_original_original['TotalCharges'] / (df_original_original['tenure'] + 1)

# Number of services
services = [
    'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 
    'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df_original_original['num_services'] = df_original_original[services].apply(lambda x: sum([1 if v in ['Yes','DSL','Fiber optic'] else 0 for v in x]), axis=1)

# Number of streaming services
df_original_original['num_streaming_services'] = df_original_original[['StreamingTV','StreamingMovies']].apply(lambda x: sum([1 if v=='Yes' else 0 for v in x]), axis=1)

# Tech support flag
df_original_original['has_tech_support'] = df_original_original['TechSupport'].apply(lambda x: 1 if x=='Yes' else 0)

# Paperless + Electronic check
df_original_original['paperless_echeck'] = ((df_original_original['PaperlessBilling']=='Yes') & (df_original_original['PaymentMethod']=='Electronic check')).astype(int)

# Long-term contract
df_original_original['is_long_contract'] = df_original_original['Contract'].apply(lambda x: 1 if x in ['One year','Two year'] else 0)

# Family status
df_original_original['family_status'] = df_original_original['Partner'] + '_' + df_original_original['Dependents']

# Tenure × Contract interaction
df_original_original['tenure_contract'] = df_original_original['tenure'].astype(str) + '_' + df_original_original['Contract']

## CATBOOST with feature eng

In [ ]:
# ===========================
# 3. Define features
# ===========================

target_col = 'Churn'

# All features except customerID and target
feature_cols = [col for col in df_original_original.columns if col not in [target_col,'customerID']]

# Original categorical features + new categorical features
categorical_cols = [
    'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
    'PaperlessBilling', 'PaymentMethod', 'tenure_group', 
    'family_status', 'tenure_contract'
]

# ===========================
# 4. Prepare X and y
# ===========================

X = df_original_original[feature_cols]
y = df_original_original[target_col].map({'No':0,'Yes':1})  # Convert to 0/1

# ===========================
# 5. Split data
# ===========================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ===========================
# 6. CatBoost Pool
# ===========================

cat_features_indices = [i for i, col in enumerate(feature_cols) if col in categorical_cols]

train_pool = Pool(X_train, y_train, cat_features=cat_features_indices)
test_pool = Pool(X_test, y_test, cat_features=cat_features_indices)

# ===========================
# 7. Train CatBoost
# ===========================

model = CatBoostClassifier(
    iterations=1000,
    depth=6,
    learning_rate=0.05,
    loss_function='Logloss',
    cat_features=cat_features_indices,
    verbose=100,
    random_state=42,
    early_stopping_rounds=50,
    use_best_model=True
)

model.fit(train_pool, eval_set=test_pool)

# ===========================
# 8. Evaluate Model
# ===========================

y_pred = model.predict(test_pool)
y_proba = model.predict_proba(test_pool)[:,1]

print("="*60)
print("CATBOOST PERFORMANCE WITH FEATURE ENGINEERING")
print("="*60)
print(f"AUC-ROC: {roc_auc_score(y_test, y_proba):.3f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Retained','Churned']))

# ===========================
# 9. Feature Importance
# ===========================

feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.get_feature_importance()
}).sort_values('importance', ascending=False)

print("\nTop 10 Most Important Features:")
print(feature_importance.head(10))

0:	learn: 0.6716693	test: 0.6717244	best: 0.6717244 (0)	total: 221ms	remaining: 3m 40s
